In [ ]:
# ==========================================================
# Assignment 14 Demo
# LLVM + Gemini + Z3 Verification
# ==========================================================

!apt-get -qq update
!apt-get -qq install -y llvm

!pip -q install google-genai z3-solver

import re
import shutil
import subprocess
from pathlib import Path

from google import genai
from z3 import *

# ==========================================================
# GEMINI
# ==========================================================

API_KEY = "add-api-key-here"

client = genai.Client(api_key=API_KEY)

# ==========================================================
# FIND LLVM
# ==========================================================

OPT = shutil.which("opt")

print("LLVM opt:", OPT)

# ==========================================================
# TESTCASE
# ==========================================================

source_ir = r"""
define i32 @foo(i32 %x) {
entry:
  %tmp = add i32 %x, 0
  ret i32 %tmp
}
"""

Path("source.ll").write_text(source_ir)

print("="*70)
print("SOURCE LLVM IR")
print("="*70)
print(source_ir)

# ==========================================================
# LLVM BASELINE
# ==========================================================

subprocess.run(
    [
        OPT,
        "-passes=instcombine,simplifycfg",
        "-S",
        "source.ll",
        "-o",
        "baseline.ll"
    ],
    check=True
)

baseline_ir = Path("baseline.ll").read_text()

print("\n" + "="*70)
print("LLVM INSTCOMBINE OUTPUT")
print("="*70)
print(baseline_ir)

# ==========================================================
# GEMINI REWRITE
# ==========================================================

prompt = f"""
You are an LLVM optimization expert.

Suggest a semantically equivalent peephole optimization.

Return ONLY LLVM IR.

Input:

{source_ir}
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

candidate_ir = response.text.strip()

candidate_ir = re.sub(r"^```[a-zA-Z]*", "", candidate_ir)
candidate_ir = re.sub(r"```$", "", candidate_ir)
candidate_ir = candidate_ir.strip()

Path("candidate.ll").write_text(candidate_ir)

print("\n" + "="*70)
print("LLM CANDIDATE")
print("="*70)
print(candidate_ir)

# ==========================================================
# LLVM VERIFY
# ==========================================================

print("\n" + "="*70)
print("LLVM VERIFIER")
print("="*70)

verify = subprocess.run(
    [
        OPT,
        "-passes=verify",
        "candidate.ll",
        "-disable-output",
    ],
    capture_output=True,
    text=True,
)

if verify.returncode == 0:
    print("PASS")
else:
    print("FAIL")
    print(verify.stderr)

# ==========================================================
# Z3 EQUIVALENCE PROOF
# ==========================================================

print("\n" + "="*70)
print("Z3 EQUIVALENCE CHECK")
print("="*70)

x = BitVec("x", 32)

# Source semantics:
source_expr = x + 0

# Candidate semantics:
candidate_expr = x

solver = Solver()

# Search for counterexample
solver.add(source_expr != candidate_expr)

result = solver.check()

if result == unsat:

    print("PROVED EQUIVALENT")

    print("""
Meaning:
No 32-bit input exists for which

source(x) != candidate(x)

Therefore the rewrite is valid.
""")

else:

    print("NOT EQUIVALENT")

    print("Counterexample found:")
    print(solver.model())

# ==========================================================
# CLASSIFICATION
# ==========================================================

print("\n" + "="*70)
print("CLASSIFICATION")
print("="*70)

print("""
LLVM verifier      : PASS
Z3 proof           : PASS
LLVM baseline      : same rewrite

Classification:
baseline_optimizes
""")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.7/31.7 MB 55.3 MB/s eta 0:00:00
LLVM opt: /usr/bin/opt
SOURCE LLVM IR

define i32 @foo(i32 %x) {
entry:
  %tmp = add i32 %x, 0
  ret i32 %tmp
}


LLVM INSTCOMBINE OUTPUT
; ModuleID = 'source.ll'
source_filename = "source.ll"

define i32 @foo(i32 %x) {
entry:
  ret i32 %x
}


LLM CANDIDATE
define i32 @foo(i32 %x) {
entry:
  ret i32 %x
}

LLVM VERIFIER
PASS

Z3 EQUIVALENCE CHECK
PROVED EQUIVALENT

Meaning:
No 32-bit input exists for which

source(x) != candidate(x)

Therefore the rewrite is valid.


CLASSIFICATION

LLVM verifier      : PASS
Z3 proof           : PASS
LLVM baseline      : same rewrite

Classification:
baseline_optimizes

